# CarbonIQ + EcoCycle: Interactive Parameter & Solver Explorer

Interactive companion notebook for:
> **CarbonIQ + EcoCycle: A Three-Tier Urban Carbon Economy and Hybrid Combinatorial Optimization Framework for Emerging Cities**

Use this notebook in **JupyterLab / Jupyter Notebook** or **Google Colab** to inspect the synthetic Accra datasets, modify penalties ($\lambda_1$ through $\lambda_{10}$), Aayalolo QBS trunk schedules, solar profiles, and annealing budgets, and compare solvers across **P1–P5** against the 25-seed exact ground-state reference files (`GS_energies_*.txt`).

### Quick Start
* **Local Jupyter:** Launch Jupyter from the repository root (`jupyter lab notebooks/CarbonIQ_Explorer.ipynb`) and run all cells.
* **Google Colab:** Upload `carboniq-ecocycle-benchmarks-v2.zip` (or clone your GitHub repo) into `/content/`. Cell 1 automatically detects Colab, unzips the archive if needed, and adds the repository root to `sys.path`.
* **Interactive Sliders:** If `ipywidgets` is installed (default in Colab and most Jupyter environments), interactive widget panels are enabled automatically in Sections 2, 3, and 5. Otherwise, edit the `PARAMS` dictionary at the top of any cell and re-run.


In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP (AUTO-DETECTS JUPYTER OR GOOGLE COLAB)
# ==============================================================================
import os
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Locate repository root whether launched from repo root, notebooks/, or Colab
def locate_repo_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/carboniq-ecocycle-benchmarks"),
    ]
    # If running in Colab and the zip was uploaded to /content, auto-extract it
    for zname in ["carboniq-ecocycle-benchmarks-v2.zip", "carboniq-ecocycle-benchmarks.zip"]:
        zpath = Path("/content") / zname
        if zpath.exists() and not Path("/content/carboniq-ecocycle-benchmarks/src").exists():
            with zipfile.ZipFile(zpath, "r") as zf:
                zf.extractall("/content")
            print(f"Auto-extracted {zpath} into /content/carboniq-ecocycle-benchmarks")

    for c in candidates:
        if (c / "src" / "qubo_builders.py").exists():
            return c.resolve()
    raise FileNotFoundError(
        "Could not locate src/qubo_builders.py. In Colab, upload the benchmark zip to /content/ "
        "or git clone the repository first."
    )

REPO_ROOT = locate_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tools.generate_synthetic_data import generate_all
from src.qubo_builders import (
    compute_mrv_credits,
    compute_street_weights,
    build_p1_routing_qubo,
    evaluate_p1_solution,
    build_p2_vpp_qubo,
    evaluate_p2_solution,
    build_p3_ev_corridor_qubo,
    evaluate_p3_solution,
    build_p4_hub_siting_qubo,
    evaluate_p4_solution,
)
from src.solvers import (
    solve_dijkstra_p5,
    solve_hungarian_p5,
    solve_exact_qubo,
    solve_exact_vpp_subspace,
    solve_simulated_annealing,
    solve_subspace_sa_vpp,
    solve_simulated_bifurcation,
    baseline_p2_uncoordinated,
    baseline_p4_nar_greedy,
)
from src.binary_vca import solve_binary_vca

try:
    import ipywidgets as widgets
    from IPython.display import display
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

VCA_DEVICE = "auto"  # Uses CUDA GPU automatically in Colab if enabled, else CPU
print(f"Repo root        : {REPO_ROOT}")
print(f"PyTorch device   : {'cuda' if torch.cuda.is_available() else 'cpu'} (VCA_DEVICE='{VCA_DEVICE}')")
print(f"ipywidgets ready : {WIDGETS_AVAILABLE}")


---
## 1. Load or Regenerate Accra Synthetic CSV Datasets

Switch `SCALE` between `"small"`, `"medium"`, and `"large"` or change `SEED` to inspect new synthetic realizations calibrated to empirical GAMA $\text{PM}_{2.5}$ medians and Aayalolo QBS trunk routes.


In [ ]:
# ==============================================================================
# CELL 2: SELECT DATASET SCALE & INSPECT CSV SCHEMAS
# ==============================================================================
SCALE = "small"      # Options: "small", "medium", "large"
SEED = 42            # Change seed to generate fresh synthetic Accra realizations
USE_PREGENERATED = True  # Set False to generate fresh data for custom SEED

if USE_PREGENERATED and SEED == 42:
    DATA_DIR = REPO_ROOT / "data" / f"starter_{SCALE}"
else:
    DATA_DIR = REPO_ROOT / "data" / f"custom_{SCALE}_seed{SEED}"
    generate_all(scale=SCALE, seed=SEED, out_dir=str(DATA_DIR))

df_road  = pd.read_csv(DATA_DIR / "accra_road_network.csv")
df_req   = pd.read_csv(DATA_DIR / "consumer_waste_requests.csv")
df_fleet = pd.read_csv(DATA_DIR / "fleet_and_collectors.csv")
df_vpp   = pd.read_csv(DATA_DIR / "vpp_appliances.csv")
df_grid  = pd.read_csv(DATA_DIR / "grid_hourly_profile.csv")
df_corr  = pd.read_csv(DATA_DIR / "corridor_demand_aayalolo.csv")
df_sites = pd.read_csv(DATA_DIR / "candidate_hub_sites.csv")

summary = pd.DataFrame([
    {"CSV File": "accra_road_network.csv",       "Rows": len(df_road),  "Columns": len(df_road.columns)},
    {"CSV File": "consumer_waste_requests.csv",  "Rows": len(df_req),   "Columns": len(df_req.columns)},
    {"CSV File": "fleet_and_collectors.csv",     "Rows": len(df_fleet), "Columns": len(df_fleet.columns)},
    {"CSV File": "vpp_appliances.csv",           "Rows": len(df_vpp),   "Columns": len(df_vpp.columns)},
    {"CSV File": "grid_hourly_profile.csv",      "Rows": len(df_grid),  "Columns": len(df_grid.columns)},
    {"CSV File": "corridor_demand_aayalolo.csv", "Rows": len(df_corr),  "Columns": len(df_corr.columns)},
    {"CSV File": "candidate_hub_sites.csv",      "Rows": len(df_sites), "Columns": len(df_sites.columns)},
])
print(f"Loaded dataset from: {DATA_DIR}")
print(summary.to_string(index=False))


---
## 2. Tier 1–2 MRV Credit Ledger & Two-Stage Anti-Fraud Gate (Equation 1)

Test how commuter modal shift across **trotros**, **Aayalolo QBS trunk buses**, and **SmartTransyt EVs** combines with two-stage weighbridge-verified waste diversion ($I_{u,g}^{(1)} I_{u,g}^{(2)}$) and smart-home eco-cycles. Toggle `hub_weighbridge_verified` to verify that unweighed gate claims mint $0.00\,\text{kg CO}_2\text{e}$.


In [ ]:
# ==============================================================================
# CELL 3: INTERACTIVE TIER 1-2 MRV CREDIT SANDBOX (EQUATION 1)
# ==============================================================================
def run_mrv_explorer(
    trotro_km=12.0,
    aayalolo_km=18.0,
    aayalolo_pax=55,
    smarttransyt_ev_km=10.0,
    organic_bsf_kg=12.0,
    pet_ecocycle_kg=6.0,
    gate_confirmed=True,
    hub_weighbridge_verified=True,
    appliance_saved_kwh=0.7,
    kappa_grid=0.74,
):
    res = compute_mrv_credits(
        transit_trips=[
            {"mode": "trotro",          "e_veh_kg_km": 0.85, "occupancy": 14,           "distance_km": trotro_km},
            {"mode": "aayalolo_qbs",    "e_veh_kg_km": 1.15, "occupancy": aayalolo_pax, "distance_km": aayalolo_km},
            {"mode": "smarttransyt_ev", "e_veh_kg_km": 0.00, "occupancy": 20,           "distance_km": smarttransyt_ev_km},
        ],
        waste_logs=[
            {"stream": "org_bsf",      "gamma_kg_co2_per_kg": 1.85, "mass_kg": organic_bsf_kg,  "gate_confirmed": gate_confirmed, "hub_weighbridge_verified": hub_weighbridge_verified},
            {"stream": "PET_ecocycle", "gamma_kg_co2_per_kg": 1.50, "mass_kg": pet_ecocycle_kg, "gate_confirmed": gate_confirmed, "hub_weighbridge_verified": hub_weighbridge_verified},
        ],
        appliance_logs=[
            {"appliance": "smart_washer", "base_kwh": 1.8, "actual_kwh": max(0.0, 1.8 - appliance_saved_kwh), "kappa_grid_kg_kwh": kappa_grid},
        ],
    )
    print(f"Two-stage gate I^(1)*I^(2) = {int(gate_confirmed) * int(hub_weighbridge_verified)}")
    print(pd.Series(res).to_string())

    fig, ax = plt.subplots(figsize=(6.2, 2.8))
    labels = ["Transit\n(Trotro/Aayalolo/EV)", "Dual-Stream Waste\n(BSF + EcoCycle)", "Smart Appliance\n(Eco-Cycle)"]
    vals = [res["transit_kg_co2e"], res["waste_kg_co2e"], res["energy_kg_co2e"]]
    bars = ax.bar(labels, vals, color=["#1b5e3a", "#0f3468", "#e07a24"], width=0.52)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.4, f"{v:.2f} kg", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.set_ylabel("Verified Credits (kg CO2e)")
    ax.set_title(f"Weekly Verified MRV Credits: Total = {res['total_cc_kg_co2e']:.2f} kg CO2e")
    ax.set_ylim(0, max(vals + [5.0]) * 1.25)
    plt.tight_layout()
    plt.show()
    return res

if WIDGETS_AVAILABLE:
    widgets.interact(
        run_mrv_explorer,
        trotro_km=widgets.FloatSlider(value=12.0, min=0, max=40, step=2, description="Trotro km"),
        aayalolo_km=widgets.FloatSlider(value=18.0, min=0, max=50, step=2, description="Aayalolo km"),
        aayalolo_pax=widgets.IntSlider(value=55, min=15, max=80, step=5, description="Aayalolo pax"),
        smarttransyt_ev_km=widgets.FloatSlider(value=10.0, min=0, max=40, step=2, description="EV bus km"),
        organic_bsf_kg=widgets.FloatSlider(value=12.0, min=0, max=40, step=1, description="Organic kg"),
        pet_ecocycle_kg=widgets.FloatSlider(value=6.0, min=0, max=30, step=1, description="PET kg"),
        gate_confirmed=widgets.Checkbox(value=True, description="Gate I^(1)"),
        hub_weighbridge_verified=widgets.Checkbox(value=True, description="Weighbridge I^(2)"),
        appliance_saved_kwh=widgets.FloatSlider(value=0.7, min=0.0, max=1.8, step=0.1, description="Saved kWh"),
        kappa_grid=widgets.FloatSlider(value=0.74, min=0.18, max=0.80, step=0.02, description="Grid kappa"),
    )
else:
    _ = run_mrv_explorer()


---
## 3. Problem P5 & Problem P1: Clean-Air Routing, Hungarian Matching, and Analytical Phase Transition $\lambda_1^\star$

Explore how distance weight $\alpha$, SLCP exhaust weight $\beta$, and black carbon share $\phi_1$ affect **P5** Clean-Air Dijkstra vs. Shortest-GPS routing, then verify the exact analytical feasibility phase transition $\lambda_1^\star = (\mathbf{W}^\top \mathbf{x}^\star)/\|\mathbf{b}\|_2^2$ and multi-collector routing (**P1**).


In [ ]:
# ==============================================================================
# CELL 4: P5 CLEAN-AIR NAVIGATION, HUNGARIAN MATCHING & P1 PHASE TRANSITION
# ==============================================================================
P1_PARAMS = {
    "n_vehicles": 2 if SCALE == "small" else 3,
    "lam_cong": 6.0,      # Corridor crowding penalty lambda_2
    "sa_sweeps": 800,
    "sb_steps": 1200,
    "seed": SEED,
}

# --- P5 Exact Polynomial Benchmarks ---
gps = solve_dijkstra_p5(df_road, use_emission_weights=False)
clean = solve_dijkstra_p5(df_road, use_emission_weights=True)
em_red = 100.0 * (gps["emission_cost"] - clean["emission_cost"]) / max(gps["emission_cost"], 1e-6)
hung = solve_hungarian_p5(df_road, df_fleet, df_req)

print(f"[P5 Navigation] Shortest-GPS: cost={gps['emission_cost']:.2f} ({gps['distance_km']:.2f} km) "
      f"vs Clean-Air Dijkstra: cost={clean['emission_cost']:.2f} ({clean['distance_km']:.2f} km) "
      f"-> -{em_red:.2f}% SLCP cost in {clean['runtime_s']*1e3:.2f} ms")
print(f"[P5 Matching  ] Greedy NN: {hung['greedy_cost']:.3f} vs Hungarian ({hung['n_pairs']} pairs): "
      f"{hung['hungarian_cost']:.3f} -> -{hung['improvement_pct']:.2f}% in {hung['runtime_s']*1e3:.2f} ms")

# --- Analytical Phase Transition Sweep on Single-Vehicle P1 (Small Scale) ---
df_road_small = pd.read_csv(REPO_ROOT / "data/starter_small/accra_road_network.csv")
clean_s = solve_dijkstra_p5(df_road_small, use_emission_weights=True)
lam1_crit = clean_s["emission_cost"] / 2.0  # ||b||_2^2 = (+1)^2 + (-1)^2 = 2
lam_grid = np.linspace(1.0, lam1_crit * 2.4, 30)
flow_res_list, em_cost_list = [], []

for lv in lam_grid:
    inst_1v = build_p1_routing_qubo(df_road_small, n_vehicles=1, lam_flow=float(lv))
    r_ex = solve_exact_qubo(inst_1v)
    ev = evaluate_p1_solution(inst_1v, r_ex.best_z)
    flow_res_list.append(np.sqrt(ev["flow_residual_sq"]))
    em_cost_list.append(ev["emission_cost"])

# --- Multi-Collector P1 Benchmark on Current SCALE ---
p1_inst = build_p1_routing_qubo(df_road, n_vehicles=P1_PARAMS["n_vehicles"], lam_cong=P1_PARAMS["lam_cong"])
p1_rows = []
if p1_inst.n_vars <= 18:
    r = solve_exact_qubo(p1_inst)
    p1_rows.append({"Solver": r.solver_name, **evaluate_p1_solution(p1_inst, r.best_z), "Runtime_s": round(r.runtime_s, 4)})
for r in [
    solve_simulated_annealing(p1_inst, n_sweeps=P1_PARAMS["sa_sweeps"], n_restarts=4, seed=P1_PARAMS["seed"]),
    solve_simulated_bifurcation(p1_inst, n_steps=P1_PARAMS["sb_steps"], n_trials=24, seed=P1_PARAMS["seed"]),
]:
    p1_rows.append({"Solver": r.solver_name, **evaluate_p1_solution(p1_inst, r.best_z), "Runtime_s": round(r.runtime_s, 4)})

print(f"\n[P1 Multi-Collector Benchmark | {p1_inst.name} | N={p1_inst.n_vars}]")
print(pd.DataFrame(p1_rows)[["Solver", "qubo_energy", "emission_cost", "feasible", "shared_corridor_edges", "Runtime_s"]].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
# Panel (a): Phase Transition
ax = axes[0]
ax.plot(lam_grid, flow_res_list, "o-", color="#c0392b", lw=2, ms=4, label=r"Flow Residual $\|Ax - b\|_2$")
ax.axvline(lam1_crit, color="#0f3468", ls="--", lw=1.6, label=rf"Analytical $\lambda_1^\star = {lam1_crit:.3f}$")
ax.set_xlabel(r"Flow Penalty Multiplier $\lambda_1$")
ax.set_ylabel(r"Flow Residual $\|Ax - b\|_2$")
ax.set_title("(a) P1 Single-Vehicle Analytical Phase Transition")
ax.legend(fontsize=8)

# Panel (b): P5 Clean-Air vs Shortest-GPS
ax2 = axes[1]
bars = ax2.bar(["Shortest-GPS\n(Baseline)", "Clean-Air Dijkstra\n(Tier 3A)"], [gps["emission_cost"], clean["emission_cost"]], color=["#c0392b", "#1b5e3a"], width=0.48)
for b, d_km in zip(bars, [gps["distance_km"], clean["distance_km"]]):
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 0.8, f"{b.get_height():.2f}\n({d_km:.1f} km)", ha="center", va="bottom", fontsize=8.5, fontweight="bold")
ax2.set_ylabel("Composite SLCP Routing Weight W(x)")
ax2.set_title(f"(b) P5 Clean-Air Gain on '{SCALE}' (-{em_red:.1f}%)")
ax2.set_ylim(0, max(gps["emission_cost"], clean["emission_cost"]) * 1.25)
plt.tight_layout()
plt.show()


---
## 4. Problem P2: Virtual Power Plant (VPP) Scheduling, Subspace Reduction & Binary VCA

Modify the midday solar target multiplier, solar tracking weight $\lambda_4$, or Binary VCA annealing stages (`vca_anneal_steps`) to compare **Uncoordinated Peak**, **Exact $6^5$ Manifold**, **Unconstrained SA**, **Subspace SA**, **SB+SRWS**, and **Binary VCA (Dilated RNN + Adam)**.


In [ ]:
# ==============================================================================
# CELL 5: P2 VPP LOAD SHIFTING & SOLVER COMPARISON (WITH OPTIONAL BINARY VCA)
# ==============================================================================
P2_PARAMS = {
    "solar_target_scale": 1.00,  # Try 1.25 for higher midday PV capacity
    "lam_solar": 0.05,           # lambda_4 solar tracking penalty
    "sa_sweeps": 900,
    "run_binary_vca": (SCALE == "small"),  # Set True on medium/large when using a Colab GPU
    "vca_anneal_steps": 80,      # Increase to 200 to observe exact ground-state convergence
    "vca_batch_size": 64,
    "seed": SEED,
}

df_grid_mod = df_grid.copy()
df_grid_mod["solar_target_kw"] = df_grid_mod["solar_target_kw"] * P2_PARAMS["solar_target_scale"]
p2_inst = build_p2_vpp_qubo(df_vpp, df_grid_mod, lam_solar=P2_PARAMS["lam_solar"])

results_p2 = []
exact_e = None
if p2_inst.metadata["n_slots"] ** p2_inst.metadata["n_app"] <= 100_000:
    r_ex = solve_exact_vpp_subspace(p2_inst)
    exact_e = r_ex.best_energy
    results_p2.append(r_ex)

results_p2.extend([
    baseline_p2_uncoordinated(p2_inst),
    solve_simulated_annealing(p2_inst, n_sweeps=P2_PARAMS["sa_sweeps"], n_restarts=4, seed=P2_PARAMS["seed"]),
    solve_subspace_sa_vpp(p2_inst, n_sweeps=600, n_restarts=4, seed=P2_PARAMS["seed"]),
    solve_simulated_bifurcation(p2_inst, n_steps=1200, n_trials=24, seed=P2_PARAMS["seed"]),
])

if P2_PARAMS["run_binary_vca"]:
    results_p2.append(
        solve_binary_vca(
            p2_inst,
            batch_size=P2_PARAMS["vca_batch_size"],
            hidden_dim=20,
            n_warmup=35,
            n_anneal=P2_PARAMS["vca_anneal_steps"],
            n_train=2,
            seed=P2_PARAMS["seed"],
            device=VCA_DEVICE,
        )
    )

rows_p2 = []
for r in results_p2:
    ev = evaluate_p2_solution(p2_inst, r.best_z)
    res_var = ((r.best_energy - exact_e) / p2_inst.n_vars) if exact_e is not None else np.nan
    rows_p2.append({
        "Solver": r.solver_name,
        "Energy": round(r.best_energy, 3),
        "Res/Var": round(res_var, 5) if not np.isnan(res_var) else None,
        "CO2_Red_%": ev["carbon_reduction_pct"],
        "Peak_Shave_%": ev["evening_peak_shaving_pct"],
        "Solar_RMSE_kW": ev["solar_rmse_kw"],
        "Runtime_s": round(r.runtime_s, 4),
    })

df_p2_res = pd.DataFrame(rows_p2)
print(f"[P2 VPP Benchmark | {p2_inst.name} | N={p2_inst.n_vars}]")
print(df_p2_res.to_string(index=False))

# Plot diurnal load profile: Uncoordinated vs Best Optimized Schedule vs Solar Target
best_r = min([r for r in results_p2 if r.solver_name != "Baseline_Uncoordinated"], key=lambda x: x.best_energy)
base_r = [r for r in results_p2 if r.solver_name == "Baseline_Uncoordinated"][0]
meta = p2_inst.metadata
Y_base = base_r.best_z.reshape(meta["n_app"], meta["n_slots"])
Y_opt  = best_r.best_z.reshape(meta["n_app"], meta["n_slots"])
load_base = meta["P"] @ Y_base
load_opt  = meta["P"] @ Y_opt
hrs = [f"{h:02d}:00" for h in meta["hours"]]

fig, axes = plt.subplots(1, 2, figsize=(11.2, 3.8))
x = np.arange(len(hrs))
w = 0.34
axes[0].bar(x - w/2, load_base, width=w, color="#c0392b", alpha=0.85, label="Uncoordinated Peak")
axes[0].bar(x + w/2, load_opt,  width=w, color="#1b5e3a", alpha=0.90, label=f"Optimized ({best_r.solver_name})")
axes[0].plot(x, meta["P_target"], "o--", color="#e67e22", lw=2.2, label="Solar Target P_bar(t)")
axes[0].set_xticks(x)
axes[0].set_xticklabels(hrs)
axes[0].set_ylabel("Flexible Neighborhood Load (kW)")
axes[0].set_title(f"(a) P2 Diurnal Load Shifting ({p2_inst.name})")
axes[0].legend(fontsize=8)

plot_df = df_p2_res[df_p2_res["Solver"] != "Baseline_Uncoordinated"]
axes[1].barh(plot_df["Solver"], plot_df["Energy"], color="#0f3468", alpha=0.85)
if exact_e is not None:
    axes[1].axvline(exact_e, color="#1b5e3a", ls="--", lw=1.6, label=f"Exact GS = {exact_e:.2f}")
    axes[1].legend(fontsize=8)
axes[1].set_xlabel("QUBO Objective Energy J_2(y) (lower is better)")
axes[1].set_title("(b) Solver Energy Comparison on P2")
plt.tight_layout()
plt.show()


---
## 5. Problem P3: SmartTransyt Electric Minibus Allocation Coordinated with Aayalolo QBS (Equation 7)

Adjust `aayalolo_supply_factor` (e.g., `0.0` = Aayalolo service disruption, `1.0` = baseline schedule, `1.25` = expanded trunk frequency) and `max_ev_buses` to inspect how residual commuter demand $D_{ct} = \max(0, D_{ct}^0 - Q_{ct}^0)$ and displaced diesel *trotro* $\text{CO}_2$ shift.


In [ ]:
# ==============================================================================
# CELL 6: P3 SMARTTRANSYT EV ALLOCATION + AAYALOLO QBS SENSITIVITY (EQ. 7)
# ==============================================================================
def run_p3_explorer(aayalolo_supply_factor=1.0, max_ev_buses=2, omega_disp=12.0):
    df_c = df_corr.copy()
    df_c["aayalolo_qbs_supply_pax_hr"] = np.round(df_c["aayalolo_qbs_supply_pax_hr"] * aayalolo_supply_factor).astype(int)
    df_c["residual_demand_pax_hr"] = np.maximum(0, df_c["total_demand_pax_hr"] - df_c["aayalolo_qbs_supply_pax_hr"])

    p3_inst = build_p3_ev_corridor_qubo(df_c, df_fleet, omega_disp=omega_disp, max_buses=max_ev_buses, use_range_slack=True)
    rows = []
    if p3_inst.n_vars <= 18:
        r_ex = solve_exact_qubo(p3_inst)
        rows.append({"Solver": r_ex.solver_name, **evaluate_p3_solution(p3_inst, r_ex.best_z), "Runtime_s": round(r_ex.runtime_s, 4)})
    for r in [
        solve_simulated_annealing(p3_inst, n_sweeps=900, n_restarts=4, seed=SEED),
        solve_simulated_bifurcation(p3_inst, n_steps=1200, n_trials=24, seed=SEED),
    ]:
        rows.append({"Solver": r.solver_name, **evaluate_p3_solution(p3_inst, r.best_z), "Runtime_s": round(r.runtime_s, 4)})

    df_res = pd.DataFrame(rows)
    print(f"[P3 | {p3_inst.name} | N={p3_inst.n_vars} | Aayalolo supply x{aayalolo_supply_factor:.2f} | "
          f"Total residual demand = {df_c['residual_demand_pax_hr'].sum()} pax/hr]")
    print(df_res[["Solver", "qubo_energy", "feasible", "displaced_diesel_co2_kg", "unmet_residual_pax_hr", "active_bus_hours", "Runtime_s"]].to_string(index=False))
    return df_res

if WIDGETS_AVAILABLE:
    max_b_avail = int((df_fleet["vehicle_type"] == "smarttransyt_ev").sum())
    widgets.interact(
        run_p3_explorer,
        aayalolo_supply_factor=widgets.FloatSlider(value=1.0, min=0.0, max=1.5, step=0.25, description="Aayalolo x"),
        max_ev_buses=widgets.IntSlider(value=min(2, max_b_avail), min=1, max=min(6, max_b_avail), step=1, description="EV Buses"),
        omega_disp=widgets.FloatSlider(value=12.0, min=0.0, max=40.0, step=4.0, description="omega_1"),
    )
else:
    _ = run_p3_explorer(aayalolo_supply_factor=1.0, max_ev_buses=2 if SCALE == "small" else 4)


---
## 6. Problem P4: Budgeted BSF Hub Siting — Soft Penalty vs. Strict-Budget Slack (Equation 8)

Compare the compact soft-penalty siting formulation ($N = L$) against the slack-augmented strict-budget formulation across **NAR Greedy**, **SA**, **SB+SRWS**, and (for `small`) **Exact Enumeration**.


In [ ]:
# ==============================================================================
# CELL 7: P4 BUDGETED BSF HUB SITING — SOFT VS STRICT-SLACK COMPARISON (EQ. 8)
# ==============================================================================
P4_PARAMS = {
    "budget_multiplier": 1.00,   # Scale municipal capital budget B
    "omega_cov": 0.35,           # Household coverage linear weight omega_2
    "omega_cap": 0.08,           # Daily waste capacity weight omega_3
    "lam_budget": 1.80,          # Budget penalty lambda_8
    "lam_cov": 0.45,             # Coverage penalty lambda_9
    "lam_hubs": 0.60,            # Hub-count penalty lambda_10
}

base_inst = build_p4_hub_siting_qubo(df_sites)
budget_custom = base_inst.metadata["budget_usd"] * P4_PARAMS["budget_multiplier"]

p4_rows = []
for strict_slack in [False, True]:
    inst = build_p4_hub_siting_qubo(
        df_sites,
        budget_usd=budget_custom,
        omega_cov=P4_PARAMS["omega_cov"],
        omega_cap=P4_PARAMS["omega_cap"],
        lam_budget=P4_PARAMS["lam_budget"],
        lam_cov=P4_PARAMS["lam_cov"],
        lam_hubs=P4_PARAMS["lam_hubs"],
        strict_budget_slack=strict_slack,
    )
    solvers = [
        baseline_p4_nar_greedy(inst),
        solve_simulated_annealing(inst, n_sweeps=900, n_restarts=4, seed=SEED),
        solve_simulated_bifurcation(inst, n_steps=1200, n_trials=24, seed=SEED),
    ]
    if inst.n_vars <= 18:
        solvers.insert(0, solve_exact_qubo(inst))
    for r in solvers:
        ev = evaluate_p4_solution(inst, r.best_z)
        p4_rows.append({
            "Formulation": "Strict-Slack" if strict_slack else "Soft-Penalty",
            "N": inst.n_vars,
            "Solver": r.solver_name,
            "Energy": ev["qubo_energy"],
            "Sites": ev["selected_sites"],
            "Budget_%": ev["budget_utilization_pct"],
            "Households": ev["households_covered"],
            "Cap_t/d": ev["daily_capacity_tonnes"],
            "Strict_Feas": ev["strict_feasible"],
            "Runtime_s": round(r.runtime_s, 4),
        })

df_p4_res = pd.DataFrame(p4_rows)
print(f"[P4 BSF Hub Siting | L={len(df_sites)} candidate sites | Budget=${budget_custom:,.0f}]")
print(df_p4_res.to_string(index=False))


---
## 7. 25-Seed Exact Ground-State Reference Inspector (`GS_energies_*.txt`) & Custom Export

Directly loads the 25-seed exact ground-state files modeled on `VectorInstitute/VariationalNeuralAnnealing`, summarizes their distributions, and exports your session results to `results/explorer_custom_run.csv`.


In [ ]:
# ==============================================================================
# CELL 8: INSPECT 25-SEED GS_energies_*.txt FILES & EXPORT CUSTOM RUN
# ==============================================================================
gs_files = {
    "P1 Routing (1v, N=8)":    REPO_ROOT / "data/P1_routing/GS_energies_p1_small_1v.txt",
    "P1 Routing (2v, N=16)":   REPO_ROOT / "data/P1_routing/GS_energies_p1_small_2v.txt",
    "P2 VPP (5x6, N=30)":      REPO_ROOT / "data/P2_vpp/GS_energies_p2_5x6.txt",
    "P3 EV+Aayalolo (N=18)":   REPO_ROOT / "data/P3_ev_corridor/GS_energies_p3_small.txt",
    "P4 Siting (14s, N=14)":   REPO_ROOT / "data/P4_hub_siting/GS_energies_p4_14sites.txt",
}

gs_summary = []
fig, axes = plt.subplots(1, 5, figsize=(13.5, 2.7))
for ax, (label, fpath) in zip(axes, gs_files.items()):
    arr = np.loadtxt(fpath, comments="#", usecols=(0, 1, 2, 3, 4))
    energies = arr[:, 1]
    feas_rate = 100.0 * np.mean(arr[:, 4])
    gs_summary.append({
        "Benchmark": label,
        "Seeds": len(energies),
        "N": int(arr[0, 3]),
        "Mean_GS": round(float(np.mean(energies)), 4),
        "Std_GS": round(float(np.std(energies)), 4),
        "Min_GS": round(float(np.min(energies)), 4),
        "Max_GS": round(float(np.max(energies)), 4),
        "Feasible_%": round(feas_rate, 1),
    })
    ax.hist(energies, bins=8, color="#0f3468", alpha=0.82, edgecolor="white")
    ax.set_title(label, fontsize=8, fontweight="bold")
    ax.set_xlabel("Exact GS J*", fontsize=7.5)
    ax.tick_params(labelsize=7)

axes[0].set_ylabel("Count (25 seeds)", fontsize=8)
plt.suptitle("25-Seed Exact Ground-State Distributions (GS_energies_*.txt)", fontsize=10, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

print(pd.DataFrame(gs_summary).to_string(index=False))

# Export combined custom P2 + P4 tables to results/explorer_custom_run.csv
out_csv = REPO_ROOT / "results" / "explorer_custom_run.csv"
df_p4_res.to_csv(out_csv, index=False)
print(f"\nExported custom P4 exploration table to: {out_csv}")
